# ML20 · NB1 · Main project
**Machine Learning I · Group Project 2026-27 · The Guild of Gentle Landings**

**Goal.** Build the model that predicts, one hour after an accident, whether it was **fatal** (`fatal = 1`). Metric: **F1 of class 1**.

**Rules for the whole notebook**
1. **Every decision traces back to NB0.** In the markdown, say which NB0 finding (section 7, "Findings and decisions for NB1") you are applying.
2. **Two kinds of preparation steps:**
   - **Deterministic rules** (no number is learned from the data: upper-casing, `0 → NaN`, out-of-range → NaN, flags, ratios) go in the `clean_*` functions of section 1. They are applied to `learn` and `predict` alike, before the CV, and cannot leak.
   - **Fitted steps** (a number is learned: imputation values, quantile caps, encodings, scaling, top-N makes) go **inside a Pipeline**, so they are fitted on the training fold only.
3. **No leakage, no `predict` peeking:** nothing is fitted on `predict`, and the threshold is chosen on out-of-fold predictions only.
4. **Reproducible:** `SEED = 5` everywhere; every step is a function or a transformer that NB9 can reuse.

**Contents**
0. Setup (Person 1)
1. Deterministic cleaning: `clean_all` (Person 1 + one function per person)
2. Validation and evaluation tools (Person 1)
3. When and where (Person 2) · 4. Weather (Person 3) · 5. Aircraft and engine (Person 4) · 6. Pilot, flight and interactions (Person 5)
7. Pipelines, baseline, feature selection and final evaluation (Person 1)


## 0. Setup
Imports, library versions (NB9 needs them to reproduce the result), `SEED`, and loading the two files from this folder with relative paths.

*Add the imports for your own models (e.g. `xgboost`, `lightgbm`) in your own section, so this cell keeps working for everyone.*


In [1]:
# ===== SHARED SETUP (all parts use this) =====
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

print(f"numpy {np.__version__} · pandas {pd.__version__} · scikit-learn {sklearn.__version__} · matplotlib {matplotlib.__version__}")

GROUP = "20"
SEED = 5
TARGET, ID = "fatal", "row_id"

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

learn = pd.read_csv("learn.csv")
predict = pd.read_csv("predict.csv")
BASE = learn[TARGET].mean()
print(f"learn {learn.shape} · predict {predict.shape} · base fatal rate {BASE:.1%}")


numpy 2.1.3 · pandas 2.2.3 · scikit-learn 1.6.1 · matplotlib 3.10.0
learn (16979, 42) · predict (4245, 41) · base fatal rate 17.0%


## 1. Deterministic cleaning: `clean_all`
One function, `clean_all(df)`, applied in the same way to `learn` and `predict`. It chains one function per group of columns. **Each person fills in their own function** (code + markdown in their section) and must not touch the others.

| Function | Owner | Columns |
|---|---|---|
| `clean_when_where` | Person 2 | `ev_year`, `ev_month`, `ev_dow`, `ev_hour`, `light_cond`, `apt_dist`, `apt_elev` |
| `clean_weather` | Person 3 | `wx_*`, `sky_*`, `vis_sm`, `wind_*`, `gust_*`, `altimeter` |
| `clean_aircraft` | Person 4 | `acft_*`, `homebuilt`, `fixed_retractable`, `ifr_equipped_cert`, `num_eng`, `total_seats`, `eng_*`, `carb_fuel_injection` (+ `seat_vintage_ratio`) |
| `clean_pilot_flight` | Person 5 | `far_part`, `type_fly`, `flt_plan_filed`, `second_pilot`, `pilot_*`, `phase` |
| `add_interactions` | Person 5 | crosses between groups, built from the **cleaned** columns |
| `global_drops` | Person 1 | columns dropped for the whole project |

**Rules for these functions:** they receive a DataFrame and return a DataFrame. Only deterministic steps: **never** compute a median, quantile, frequency or target mean here (that goes in the Pipeline). Flags such as `apt_elev_missing` are created **here, before** any imputation. Do not drop or reorder rows.

**Global drops (Person 1, from NB0):**
- `ev_country`: a single value, `USA` (NB0 1.1).
- `wx_temp_c`: an exact conversion of `wx_temp` (NB0 1.1 / 1.6).
- `gust_ind`: practically inverted with respect to `gust_kts` (NB0 1.4, Part 3). It is dropped **after** `clean_weather`, so Person 3 can still use it to build `gust_present`.

NB0 found no duplicated rows and no duplicated `row_id`; `clean_all` checks this again so a later change can't break it.


In [2]:
# ----- one function per person: replace the body, keep the name and the signature -----
def clean_when_where(df: pd.DataFrame) -> pd.DataFrame:
    """Person 2 · deterministic rules for the when-and-where columns."""
    return df


def clean_weather(df: pd.DataFrame) -> pd.DataFrame:
    """Person 3 · deterministic rules for the weather columns (build gust_present here)."""
    return df


def clean_aircraft(df: pd.DataFrame) -> pd.DataFrame:
    """Person 4 · deterministic rules for the aircraft and engine columns (+ seat_vintage_ratio)."""
    return df


def clean_pilot_flight(df: pd.DataFrame) -> pd.DataFrame:
    """Person 5 · deterministic rules for the pilot and flight columns."""
    return df


def add_interactions(df: pd.DataFrame) -> pd.DataFrame:
    """Person 5 · interactions between groups, built from the cleaned columns."""
    return df


# ----- Person 1 -----
GLOBAL_DROP = ["ev_country", "wx_temp_c", "gust_ind"]


def global_drops(df: pd.DataFrame) -> pd.DataFrame:
    """Columns dropped for the whole project (see the markdown above)."""
    return df.drop(columns=[c for c in GLOBAL_DROP if c in df.columns])


def clean_all(df: pd.DataFrame) -> pd.DataFrame:
    """All deterministic rules, in order. Works on a copy; the same function is used for learn and predict."""
    n, ids = len(df), df[ID].copy()
    out = df.copy()
    for step in (clean_when_where, clean_weather, clean_aircraft, clean_pilot_flight, add_interactions, global_drops):
        out = step(out)
    # safety checks: no row lost, reordered or duplicated
    assert len(out) == n and out[ID].equals(ids), "a cleaning step changed the rows"
    assert not out[ID].duplicated().any(), "duplicated row_id"
    return out


In [3]:
learn_c = clean_all(learn)
predict_c = clean_all(predict)

X = learn_c.drop(columns=[ID, TARGET])
y = learn_c[TARGET]
X_predict = predict_c.drop(columns=[ID])

assert list(X.columns) == list(X_predict.columns), "learn and predict must end with the same columns"
print(f"X {X.shape} · X_predict {X_predict.shape}")
print(f"numeric: {X.select_dtypes('number').shape[1]} · categorical: {X.select_dtypes(exclude='number').shape[1]}")


X (16979, 37) · X_predict (4245, 37)
numeric: 19 · categorical: 18


**Note.** Re-run this cell (and everything below it) whenever a `clean_*` function changes. The assert guarantees that `learn` and `predict` always end with the same columns, which NB9 needs.


## 2. Validation and evaluation tools
**Validation scheme (NB0 1.5: `learn` and `predict` come from the same distribution, a random split of 2008–2023):**
- A **stratified holdout of 20%** (`X_hold`, `y_hold`) is set aside **now** and not used until the final evaluation (section 7). It gives one honest score for the final model and threshold.
- Everything else (model comparison, tuning, threshold choice, feature selection) uses **stratified 5-fold CV** on the remaining 80% (`X_dev`, `y_dev`). Stratified, so each fold keeps the ~17% fatal rate.

**Threshold.** F1 of class 1 depends on the decision threshold, and 0.5 is rarely the best when only 17% of cases are positive. `evaluate` collects the **out-of-fold** probabilities (each row predicted by a model that did not see it) and searches the threshold on them. The holdout is never used for that.


In [4]:
X_dev, X_hold, y_dev, y_hold = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(f"dev {X_dev.shape[0]:,} rows (fatal {y_dev.mean():.1%}) · holdout {X_hold.shape[0]:,} rows (fatal {y_hold.mean():.1%})")


dev 13,583 rows (fatal 17.0%) · holdout 3,396 rows (fatal 17.0%)


In [5]:
RESULTS = {}   # every evaluate() call adds one row here; section 7 turns it into the comparison table


def find_best_threshold(y_true, proba, grid=np.arange(0.05, 0.951, 0.01)):
    """Threshold on the predicted probability that maximises F1 of class 1. Use it on out-of-fold probabilities only."""
    scores = [f1_score(y_true, (proba >= t).astype(int), zero_division=0) for t in grid]
    best = int(np.argmax(scores))
    return float(grid[best]), float(scores[best])


def evaluate(model, name, X_=None, y_=None, cv=CV, show=True):
    """Stratified K-fold CV of a full pipeline on the dev set.

    Reports F1 / precision / recall of class 1 at threshold 0.5 and at the best out-of-fold threshold,
    prints the confusion matrix at the best threshold and stores everything in RESULTS[name].
    Returns the out-of-fold probabilities.
    """
    X_ = X_dev if X_ is None else X_
    y_ = y_dev if y_ is None else y_
    t0 = time.time()
    proba = cross_val_predict(clone(model), X_, y_, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    secs = time.time() - t0

    t_best, _ = find_best_threshold(y_, proba)
    row = {}
    for label, t in (("@0.5", 0.5), ("@best", t_best)):
        pred = (proba >= t).astype(int)
        row[f"F1{label}"] = f1_score(y_, pred, zero_division=0)
        row[f"precision{label}"] = precision_score(y_, pred, zero_division=0)
        row[f"recall{label}"] = recall_score(y_, pred, zero_division=0)
    row["threshold"] = t_best
    row["cv_seconds"] = secs
    RESULTS[name] = row

    if show:
        cm = confusion_matrix(y_, (proba >= t_best).astype(int))
        print(f"{name}: F1 = {row['F1@best']:.3f} at threshold {t_best:.2f} (F1 at 0.5 = {row['F1@0.5']:.3f}) · {secs:.1f}s")
        display(pd.DataFrame(cm, index=["true 0", "true 1"], columns=["pred 0", "pred 1"]))
    return proba


def results_table():
    """All evaluated models, best F1 first."""
    return pd.DataFrame(RESULTS).T.sort_values("F1@best", ascending=False).round(3)


**Note.** The F1 at the best threshold is slightly optimistic, because the threshold was chosen on the same out-of-fold predictions it is scored on. That is fine for **comparing** models (they all get the same advantage). The honest number for the final model comes from the holdout in section 7.


### Temporary preprocessors (v0)
So that everyone can test models before the pipelines are assembled, here are two **simple, temporary** preprocessors matching the two variants of the plan. In section 7 Person 1 replaces them with the real ones built from every person's transformers. Both are fitted inside the CV (no leakage).
- `make_prep_native()`: for boosting / trees. Numbers stay as they are (NaN kept), categories → integer codes (unknown or missing → −1).
- `make_prep_linear()`: for linear models, kNN, SVM. Median imputation + scaling for numbers; most-frequent imputation + one-hot for categories (unknown categories ignored).


In [6]:
num_cols = make_column_selector(dtype_include="number")
cat_cols = make_column_selector(dtype_exclude="number")


def make_prep_native():
    return ColumnTransformer([
        ("num", "passthrough", num_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                               encoded_missing_value=-1), cat_cols),
    ])


def make_prep_linear():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore", min_frequency=30)), cat_cols),
    ])


### Check that the tools work
A dummy model that always predicts "fatal" must give F1 ≈ 0.29 (NB0 1.2). It is only here to test `evaluate`; the real baseline is in section 7.


In [7]:
_ = evaluate(make_pipeline(make_prep_native(), DummyClassifier(strategy="constant", constant=1)), "check · always 1")
results_table()


check · always 1: F1 = 0.291 at threshold 0.05 (F1 at 0.5 = 0.291) · 1.9s


,pred 0,pred 1
true 0,0,11272
true 1,0,2311


,F1@0.5,precision@0.5,recall@0.5,F1@best,precision@best,recall@best,threshold,cv_seconds
check · always 1,0.291,0.17,1.0,0.291,0.17,1.0,0.05,1.919


**How to use it in your section**
```python
from sklearn.linear_model import LogisticRegression
model = make_pipeline(make_prep_linear(), LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED))
oof = evaluate(model, "P5 · logistic v1")
results_table()
```
Name your runs `"P<person> · <model> <version>"` so the final table shows who did what.


# 3. When and where (Person 2)

# 4. Weather (Person 3)

# 5. Aircraft and engine (Person 4)

# 6. Pilot, flight and interactions (Person 5)

# 7. Pipelines, baseline, feature selection and final evaluation (Person 1)